# Sprint 2 · Análise Exploratória (EDA) — Previsão de Resultados de Futebol

**Objetivo desta EDA:** antes de qualquer modelo, precisamos *entender os dados*.
O dataset traz +48 mil jogos internacionais desde 1872. Aqui vamos:

1. Carregar `results.csv` e conhecer sua estrutura (tipos, tamanho, nulos).
2. Derivar o **alvo** `resultado` (`casa` / `empate` / `fora`) a partir do placar.
   > ⚠️ O placar só existe **depois** do jogo — ele serve para criar o alvo, mas
   > **nunca** pode virar *feature* (isso seria *data leakage*).
3. Investigar 3 perguntas de negócio via gráficos:
   - **A)** Qual a proporção casa vs empate vs fora? (mede a vantagem de mando — baseline)
   - **B)** Como a média de gols evoluiu ao longo das décadas?
   - **C)** Quais as 15 seleções com mais vitórias?

Os gráficos já vêm com título e eixos prontos. As **análises e alguns trechos**
ficam marcados com `# TODO Pedro:` — é onde você completa para aprender fazendo.

In [ ]:
# Imports + estilo dos gráficos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo consistente para todos os gráficos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

pd.set_option("display.max_columns", None)
print("numpy", np.__version__, "| pandas", pd.__version__)

## 1. Carregar os dados e conhecer a estrutura

In [ ]:
# Caminho relativo à raiz do projeto (o notebook roda em notebooks/)
RAW_PATH = "../data/raw/results.csv"

df = pd.read_csv(RAW_PATH, parse_dates=["date"])
print(f"Formato: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()

In [ ]:
df.info()

In [ ]:
# describe(include="all") mostra estatísticas de numéricas E categóricas
df.describe(include="all")

## 2. Nulos e tipos de coluna

Antes de modelar precisamos saber: há valores faltando? Quais colunas são
numéricas e quais são categóricas?

In [ ]:
# Contagem de nulos por coluna
nulos = df.isna().sum()
print("Nulos por coluna:")
print(nulos[nulos > 0] if nulos.any() else "Nenhum nulo 🎉")

# TODO Pedro: se houver nulos, decida a estratégia (dropar? preencher?) e justifique.

In [ ]:
# Classificar colunas em numéricas vs categóricas
numericas = df.select_dtypes(include="number").columns.tolist()
categoricas = df.select_dtypes(include=["object", "bool"]).columns.tolist()

print("Numéricas:  ", numericas)
print("Categóricas:", categoricas)

# TODO Pedro: a coluna 'date' é datetime, não caiu em nenhum grupo acima.
#             Como você pretende usar a data como feature? (ex.: ano, mês, década)

## 3. Derivar o alvo (target)

O modelo aprende a prever `resultado`. Ele vem do placar — que **só conhecemos
depois do jogo**. Serve para criar o alvo, nunca como feature.

In [ ]:
# Deriva 'resultado' do ponto de vista do MANDANTE:
#   home_score > away_score -> 'casa'
#   home_score < away_score -> 'fora'
#   empate                  -> 'empate'
condicoes = [
    df["home_score"] > df["away_score"],
    df["home_score"] < df["away_score"],
]
df["resultado"] = np.select(condicoes, ["casa", "fora"], default="empate")

df["resultado"].value_counts()

## Gráfico A — Proporção casa vs empate vs fora

Mostra a **vantagem de mando** e serve de **baseline** ("sempre casa").

In [ ]:
# TODO Pedro: calcule a proporção (%) de cada resultado.
#             Dica: df["resultado"].value_counts(normalize=True)
prop = None  # <-- substitua

fig, ax = plt.subplots()
if prop is not None:
    prop.mul(100).plot(kind="bar", ax=ax, color=["#2a9d8f", "#e9c46a", "#e76f51"])
ax.set_title("Proporção de resultados (ponto de vista do mandante)")
ax.set_xlabel("Resultado")
ax.set_ylabel("Percentual dos jogos (%)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

# TODO Pedro: qual % a baseline "sempre casa" acertaria? Escreva a conclusão aqui.

## Gráfico B — Média de gols por década

O futebol ficou mais ofensivo ou mais defensivo ao longo do tempo?

In [ ]:
# Total de gols por jogo
df["total_gols"] = df["home_score"] + df["away_score"]

# Década a partir do ano da partida
df["decada"] = (df["date"].dt.year // 10) * 10

# TODO Pedro: agrupe por 'decada' e calcule a MÉDIA de 'total_gols'.
#             Dica: df.groupby("decada")["total_gols"].mean()
gols_por_decada = None  # <-- substitua

fig, ax = plt.subplots()
if gols_por_decada is not None:
    gols_por_decada.plot(marker="o", ax=ax)
ax.set_title("Média de gols por partida ao longo das décadas")
ax.set_xlabel("Década")
ax.set_ylabel("Média de gols por jogo")
plt.tight_layout()
plt.show()

# TODO Pedro: descreva a tendência. Alguma década fora da curva? Por quê?

## Gráfico C — Top 15 seleções por número de vitórias

In [ ]:
# Descobre o time vencedor de cada jogo (empate -> NaN, não conta)
def time_vencedor(row):
    if row["home_score"] > row["away_score"]:
        return row["home_team"]
    if row["home_score"] < row["away_score"]:
        return row["away_team"]
    return np.nan

df["vencedor"] = df.apply(time_vencedor, axis=1)

# TODO Pedro: conte as vitórias por seleção e pegue as 15 maiores.
#             Dica: df["vencedor"].value_counts().head(15)
top15 = None  # <-- substitua

fig, ax = plt.subplots(figsize=(10, 7))
if top15 is not None:
    top15.sort_values().plot(kind="barh", ax=ax, color="#264653")
ax.set_title("Top 15 seleções por número de vitórias (1872–hoje)")
ax.set_xlabel("Número de vitórias")
ax.set_ylabel("Seleção")
plt.tight_layout()
plt.show()

# TODO Pedro: esse ranking é 'justo'? Times que jogaram mais partidas levam vantagem.
#             Pense em normalizar por número de jogos — vira ideia de feature na Sprint 3.